In [ ]:
!pip install crewai


In [ ]:
!pip install crewai-tools

In [ ]:
from google.colab import userdata
serperapi=userdata.get('serperkey')

**step 1**

## import the packages

In [ ]:
import os
from crewai import Agent,Task,Crew,Process,LLM
from crewai_tools import SerperDevTool

in vs code we hide keys using .env in google colab we hide it using keys

In [ ]:
from google.colab import userdata
serper_key=userdata.get('serperkey')
os.environ["serper_api_key"]=serper_key

## **step 3 initialize the model**

In [ ]:
from google.colab import userdata
gemini_key=userdata.get('geminikey')
os.environ["gemini_api_key"]=gemini_key

In [ ]:
from re import VERBOSE
#initialize the search tool (to fetch real time travle info)
#gemini_key-'apijey'

search_tool=SerperDevTool()
llm=LLM(model="gemini/gemini-3.5-flash",
        verbose=True,
        temperature=0.5,
        api_key=gemini_key)
llm

In [ ]:
#travel researcher agent (finds historical sites + weather)
researcher=Agent(
    role="Travel Researcher",
    goal="""Find historical sites,public transport hotels,
    and real time weather for {destination}.""",
    verbose=True,
    memory=True,
    backstory="You re an expert travel researcher , providing up-to-dste information about history",
    llm=llm,
    tools=[search_tool],
    allow_delegation=True
)

# budget planner agent (ensures trip stays under $1500)

budget_planner=Agent(
    role="Budget Planner",
    goal="Find budget flights , hotels , and activities with in {budget} for {destination}",
    verbose=True,
    memory=True,
    backstory="You are a skilled budget analyst ensuring trips fit with in financial constraints",
    llm=llm,
    tools=[search_tool],
    allow_delegation=False
)

# itinerary planner agent (creates a balanced 3 day plan)
itinerary_planner=Agent(
    role="Itinerary Planner",
    goal="Create a 3day itinerary for {destination} , ensuring all historical sites are covered under {budget}",
    verbose=True,
    memory=True,
    backstory="you are an expert in trip planning , ensuring , travels get the best experience within their budget",
    llm=llm,
    tools=[search_tool],
    allow_delegation=False
)

STEP 5 Assign the task

Assign the task

In [ ]:
research_task=Task(
    description="""Find the best historical sites weather forecast,
    and public transport hotels for {destination}""",
    expected_output="""A list of top historical sites , a real time weather update,
    and 3 hotel options near public transport""",
    tools=[search_tool],
    agent=researcher
)
# budget estimation task
budget_task=Task(
    description="""Find budget , hotel options ,and daily food/transport costs for {destination}
    ensure total cost stays under {budget}""",
    expected_output="""A full cost breakdown (flights,hotel,food,attractions)
    ensuring a $1500 budget is maintained""",
    tools=[search_tool],
    agent=budget_planner
)

itinerary_task=Task(
    description="""Plan a 3 day itinerary for {destination}, focusing on historical sites,
    budget constraints , and real time weather conditions""",
    expected_output="""A detailed 3 day plan , considering weather and budget constraints,
    with transport recommendations""",
    tools=[search_tool],
    agent=itinerary_planner
)


In [ ]:
# itinerary_task=Task(
#     description="""Plan a 3 day itinerary for {destination}, focusing on historical sites,
#     budget constraints , and real time weather conditions""",
#     expected_output="""A detailed 3 day plan , considering weather and budget constraints,
#     with transport recommendations""",
#     tools=[search_tool],
#     agent=itinerary_planner
# )

STEP **6**

# step 6 set up crew



1.   what are the agents we need to provide
2.   what are the tasks also we need to provide
3. process sequential or parallel



what are the agents we need to provide


In [ ]:
crew=Crew(
    agents=[researcher,budget_planner,itinerary_planner],
            tasks=[research_task,budget_task,itinerary_task],
            process=Process.sequential
)
crew

step 7 kickoff the crew

In [ ]:
result=await crew.kickoff_async(inputs={'destination':'London','budget':'1500'})
print(result)